In [1]:
import time
import pandas as pd
import numpy as np
from typing import Callable, Any

import requests
from requests import Response
from bs4 import BeautifulSoup


In [12]:

#SHOOTING
class HTMLScraper:
    """Class for scraping the web"""

    proxy: str | None = None
    timeout: int = 10  # seconds to wait before giving up on a request
    max_retries: int = 3
    retry_backoff: int = 15  # base seconds to wait before retrying

    def scrape(self, url: str, parser_fun: Callable) -> Any:
        page = self.get_page(url)
        soup = BeautifulSoup(page.text, "html.parser")
        return parser_fun(soup)

    def get_page(self, url: str) -> Response:
        """
        Sends a GET request with a timeout, and retries with backoff
        if rate-limited or blocked (429/403) or the connection fails.
        """

        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/124.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
            "Accept-Language": "en-US,en;q=0.9",
        }

        for attempt in range(1, self.max_retries + 1):
            try:
                with requests.Session() as session:
                    page = session.get(
                        url,
                        proxies=self._proxies(),
                        headers=headers,
                        timeout=self.timeout,
                    )

                if page.status_code in (429, 403):
                    wait = int(
                        page.headers.get("Retry-After", self.retry_backoff * attempt)
                    )
                    print(f"Blocked ({page.status_code}) on {url}, waiting {wait}s (attempt {attempt})")
                    time.sleep(wait)
                    continue

                page.raise_for_status()
                return page

            except requests.exceptions.RequestException as e:
                print(f"Request failed for {url}: {e} (attempt {attempt})")
                if attempt == self.max_retries:
                    raise
                time.sleep(self.retry_backoff * attempt)

        raise RuntimeError(f"Exhausted retries fetching {url}")

    def _proxies(self) -> dict | None:
        """If the object is initialized with a proxy then use it"""
        if self.proxy:
            return {"https": self.proxy, "http": self.proxy}
        return None


In [4]:

from dataclasses import dataclass
from bs4 import BeautifulSoup, Comment


@dataclass
class BaskRefRatingsScraper(HTMLScraper):
    """Scrapes the league-wide team ratings table."""

    base_url: str = "https://www.basketball-reference.com"

    def get_team_ratings(self, season_year: int) -> list[dict]:
        url = f"{self.base_url}/leagues/NBA_{season_year}_ratings.html"
        return self.scrape(url, self._parse_ratings_table)

    def debug_list_tables(self, season_year: int) -> dict:
        """
        Diagnostic helper: returns every table id found, both directly
        in the DOM and hidden inside HTML comments. Run this first if
        the real parser can't find the table.
        """
        url = f"{self.base_url}/leagues/NBA_{season_year}_ratings.html"

        def _list_ids(page: BeautifulSoup) -> dict:
            direct_ids = [t.get("id") for t in page.find_all("table")]

            comment_ids = []
            for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
                if "<table" in comment:
                    inner = BeautifulSoup(comment, "html.parser")
                    comment_ids.extend(
                        t.get("id") for t in inner.find_all("table")
                    )

            return {"direct": direct_ids, "in_comments": comment_ids}

        return self.scrape(url, _list_ids)

    # private functions

    def _parse_ratings_table(self, page: BeautifulSoup) -> list[dict]:
        table = page.select_one("#ratings") or page.select_one(
            "table[id*=ratings]"
        )

        if table is None:
            table = self._find_table_in_comments(page, "ratings")

        if table is None:
            raise ValueError(
                "Could not locate the ratings table. "
                "Run debug_list_tables() to see available table ids."
            )

        rows = []
        for tr in table.select("tbody tr"):
            if tr.get("class") and "thead" in tr.get("class"):
                continue

            row = {}
            for cell in tr.select("td, th"):
                stat = cell.get("data-stat")
                if stat:
                    row[stat] = cell.text.strip()
            if row:
                rows.append(row)

        return rows

    @staticmethod
    def _find_table_in_comments(page: BeautifulSoup, id_substring: str):
        for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in comment and id_substring in comment:
                inner = BeautifulSoup(comment, "html.parser")
                found = inner.select_one(f"table[id*={id_substring}]")
                if found is not None:
                    return found
        return None


team_rating_data = pd.DataFrame(columns=['ranker',
                                         'team_name',
                                         'conf_id',
                                         'div_id',
                                         'wins',
                                         'losses',
                                         'win_loss_pct',
                                         'mov',
                                         'off_rtg',
                                         'def_rtg',
                                         'net_rtg',
                                         'mov_adj',
                                         'off_rtg_adj',
                                         'def_rtg_adj',
                                         'net_rtg_adj'])
team_rating_data
import time

seasons = [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017,
           2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]
scraper = BaskRefRatingsScraper()
shooting_data_list = []
for season in seasons:
    print(season)
    rows = scraper.get_team_ratings(season)
    for row in rows:
        row['season'] = season
    team_rating_data = pd.concat([team_rating_data, pd.DataFrame(rows)])
    time.sleep(4)
team_rating_data
team_rating_data_new = team_rating_data
team_rating_data_new['off_rtg'] = team_rating_data_new['off_rtg'].astype(float)
mean_off_rtg = team_rating_data_new.groupby('season').agg(off_rtg_lg_avg=('off_rtg', 'mean'),
                                                          off_rtg_lg_stdev=('off_rtg', 'std')).reset_index()
team_rating_data_new = pd.merge(team_rating_data_new, mean_off_rtg, how='left', on='season')
team_rating_data_new['off_rtg_vs_lg_avg'] = (team_rating_data_new['off_rtg'] - team_rating_data_new['off_rtg_lg_avg']) / \
                                            team_rating_data_new['off_rtg_lg_stdev']
team_rating_data_new['season'] = team_rating_data_new['season'].astype(str).str[:4]
team_rating_data_new

2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


,ranker,team_name,conf_id,div_id,wins,losses,win_loss_pct,mov,off_rtg,def_rtg,net_rtg,mov_adj,off_rtg_adj,def_rtg_adj,net_rtg_adj,season,off_rtg_lg_avg,off_rtg_lg_stdev,off_rtg_vs_lg_avg
0,1,Los Angeles Lakers,W,P,67,15,.817,8.55,107.85,98.79,9.05,8.41,107.66,98.75,8.92,2000,104.806897,3.153387,0.965027
1,2,Portland Trail Blazers,W,P,59,23,.720,6.40,108.75,101.67,7.08,6.37,108.71,101.68,7.03,2000,104.806897,3.153387,1.250434
2,3,San Antonio Spurs,W,M,53,29,.646,5.94,105.83,99.45,6.38,5.93,105.67,99.30,6.37,2000,104.806897,3.153387,0.324446
3,4,Phoenix Suns,W,P,53,29,.646,5.22,105.04,99.66,5.38,5.24,104.90,99.48,5.41,2000,104.806897,3.153387,0.073922
4,5,Utah Jazz,W,M,55,27,.671,4.46,107.97,103.09,4.88,4.52,108.00,103.06,4.94,2000,104.806897,3.153387,1.003081
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800,26,Indiana Pacers,E,C,19,63,.232,-7.99,111.01,118.97,-7.96,-7.64,111.16,118.76,-7.60,2026,115.843333,3.359319,-1.438784
801,27,Utah Jazz,W,NW,22,60,.268,-8.43,114.18,122.39,-8.21,-8.04,114.44,122.28,-7.84,2026,115.843333,3.359319,-0.495140
802,28,Sacramento Kings,W,P,22,60,.268,-10.00,111.56,121.63,-10.07,-9.58,111.70,121.34,-9.64,2026,115.843333,3.359319,-1.275060
803,29,Brooklyn Nets,E,A,20,62,.244,-9.99,108.84,119.11,-10.27,-9.70,109.00,118.96,-9.97,2026,115.843333,3.359319,-2.084748


In [5]:
team_rating_data_new.to_csv('NBA_team_rating_data_2000-2026.csv')

In [ ]:
player_epm_team = pd.merge(player_season_epm,team_dim,how='left',left_on='team_alias',right_on='team_alias')
player_epm_team_rating = pd.merge(player_epm_team,team_rating_data_new,how='left',left_on=['team','season_abbrev'],right_on=['team_name','season'])
player_epm_team_rating['off_epm_above_3'] = np.where(player_epm_team_rating['off'] >= 3,1,0)
player_epm_team_rating['usg'] = player_epm_team_rating['usg'].round(3)
player_epm_team_rating['off_rtg_vs_lg_avg'] = player_epm_team_rating['off_rtg_vs_lg_avg'].round(3)
player_epm_team_rating['usg_mp'] = player_epm_team_rating['usg'] * player_epm_team_rating['mp']
player_epm_team_rating = player_epm_team_rating.dropna(subset='off_rtg_vs_lg_avg')
player_epm_team_rating

In [ ]:
player_epm_team_rating_filter = player_epm_team_rating[player_epm_team_rating['mp'] >= 901]
pd.plotting.scatter_matrix(player_epm_team_rating_filter[['usg_mp','off_epm_above_3','off_rtg_vs_lg_avg']])

In [ ]:
import statsmodels.api as sm

X = player_epm_team_rating_filter[['usg','tot']]
y = player_epm_team_rating_filter['off_rtg_vs_lg_avg']

# Statsmodels does not add an intercept (constant) automatically
X_with_constant = sm.add_constant(X)

# Fit Ordinary Least Squares (OLS) model
model = sm.OLS(y, X_with_constant).fit()

# Print the comprehensive summary table
print(model.summary())

In [ ]:
from statsmodels.stats.outliers_influence \
import variance_inflation_factor as VIF
vals = [VIF(player_epm_team_rating_filter[['usg','tot']], i)
for i in range(0, player_epm_team_rating_filter[['usg','tot']].shape[1])]
vif = pd.DataFrame({'vif':vals},
index=player_epm_team_rating_filter[['usg','tot']].columns[0:])
vif

In [7]:
from dataclasses import dataclass
from bs4 import BeautifulSoup, Comment
#TOTALS
@dataclass
class BaskRefTotalsScraper(HTMLScraper):
    """Scrapes the league-wide player totals table."""

    base_url: str = "https://www.basketball-reference.com"

    def get_player_totals(self, season_year: int) -> list[dict]:
        url = f"{self.base_url}/leagues/NBA_{season_year}_totals.html"
        return self.scrape(url, self._parse_totals_table)

    def debug_list_tables(self, season_year: int) -> dict:
        """
        Diagnostic helper: returns every table id found, both directly
        in the DOM and hidden inside HTML comments. Run this first if
        the real parser can't find the table.
        """
        url = f"{self.base_url}/leagues/NBA_{season_year}_totals.html"

        def _list_ids(page: BeautifulSoup) -> dict:
            direct_ids = [t.get("id") for t in page.find_all("table")]

            comment_ids = []
            for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
                if "<table" in comment:
                    inner = BeautifulSoup(comment, "html.parser")
                    comment_ids.extend(
                        t.get("id") for t in inner.find_all("table")
                    )

            return {"direct": direct_ids, "in_comments": comment_ids}

        return self.scrape(url, _list_ids)

    # private functions

    def _parse_totals_table(self, page: BeautifulSoup) -> list[dict]:
        # bref has used both "totals_stats" and "totals" as the id over the years
        table = (
            page.select_one("#totals_stats")
            or page.select_one("#totals")
            or page.select_one("table[id*=totals]")
        )

        if table is None:
            table = self._find_table_in_comments(page, "totals")

        if table is None:
            raise ValueError(
                "Could not locate the totals table. "
                "Run debug_list_tables() to see available table ids."
            )

        rows = []
        for tr in table.select("tbody tr"):
            # skip the repeated in-table header rows
            if tr.get("class") and ("thead" in tr.get("class")):
                continue

            row = {}
            for cell in tr.select("td, th"):
                stat = cell.get("data-stat")
                if stat:
                    row[stat] = cell.text.strip()
            if row:
                rows.append(row)

        return rows

    @staticmethod
    def _find_table_in_comments(page: BeautifulSoup, id_substring: str):
        for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in comment and id_substring in comment:
                inner = BeautifulSoup(comment, "html.parser")
                found = inner.select_one(f"table[id*={id_substring}]")
                if found is not None:
                    return found
        return None

player_totals_data = pd.DataFrame()

seasons = [2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012,
           2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

totals_scraper = BaskRefTotalsScraper()
for season in seasons:
    print(season)
    rows = totals_scraper.get_player_totals(season)
    for row in rows:
        row['season'] = season
    player_totals_data = pd.concat([player_totals_data, pd.DataFrame(rows)])
    time.sleep(4)

player_totals_data


2000
2001
2002
2003
2004
2005
2006
2007
2008
2009
2010
2011
2012
2013
2014
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024
2025
2026


,ranker,name_display,age,team_name_abbr,pos,games,games_started,mp,fg,fga,...,trb,ast,stl,blk,tov,pf,pts,tpl_dbl,awards,season
0,1,Shaquille O'Neal,27,LAL,C,79,79,3163,956,1665,...,1078,299,36,239,223,255,2344,0,"MVP-1,DPOY-2,AS,NBA1,DEF2",2000
1,2,Vince Carter,23,TOR,SF,82,82,3126,788,1696,...,476,322,110,92,178,263,2107,1,"MVP-10,AS,NBA3",2000
2,3,Karl Malone,36,UTA,PF,82,82,2947,752,1476,...,779,304,79,71,231,229,2095,0,"MVP-4,AS,NBA2",2000
3,4,Allen Iverson,24,PHI,SG,70,70,2853,729,1733,...,267,328,144,5,230,162,1989,0,"MVP-7,AS,NBA2",2000
4,5,Gary Payton,31,SEA,PG,82,82,3425,747,1666,...,529,732,153,18,224,178,1982,2,"MVP-6,DPOY-5,AS,NBA1,DEF1",2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
729,579,Darius Brown II,26,CLE,SG,1,0,3,0,1,...,1,0,0,0,0,0,0,0,,2026
730,580,Noa Essengue,19,CHI,PF,2,0,6,0,3,...,0,0,1,0,0,1,0,0,,2026
731,581,Tosan Evbuomwan,24,NYK,SF,5,0,8,0,1,...,2,0,0,0,0,1,0,0,,2026
732,582,Chucky Hepburn,22,TOR,PG,2,0,13,0,6,...,1,2,1,0,1,2,0,0,,2026


In [8]:
player_totals_data.to_csv('NBA_player_totals_2000-2026.csv')

In [5]:
#TEAM ROSTERS
from dataclasses import dataclass
from bs4 import BeautifulSoup, Comment


@dataclass
class BaskRefRosterScraper(HTMLScraper):
    """Scrapes a single team's roster page for a given season."""

    base_url: str = "https://www.basketball-reference.com"

    # basketball-reference uses its own tricodes for some teams, differing
    # from the NBA's official ones
    TEAM_TRICODES = [
        'ATL', 'BOS', 'BRK', 'CHO', 'CHI', 'CLE', 'DAL', 'DEN', 'DET',
        'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN',
        'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHO', 'POR', 'SAC', 'SAS',
        'TOR', 'UTA', 'WAS',
    ]

    def get_roster(self, team: str, season_year: int) -> list[dict]:
        url = f"{self.base_url}/teams/{team}/{season_year}.html"
        rows = self.scrape(url, self._parse_roster_table)
        for row in rows:
            row['team'] = team
            row['season'] = season_year
        return rows

    def debug_list_tables(self, team: str, season_year: int) -> dict:
        """
        Diagnostic helper: returns every table id found, both directly
        in the DOM and hidden inside HTML comments. Run this first if
        the real parser can't find the roster table.
        """
        url = f"{self.base_url}/teams/{team}/{season_year}.html"

        def _list_ids(page: BeautifulSoup) -> dict:
            direct_ids = [t.get("id") for t in page.find_all("table")]
            comment_ids = []
            for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
                if "<table" in comment:
                    inner = BeautifulSoup(comment, "html.parser")
                    comment_ids.extend(t.get("id") for t in inner.find_all("table"))
            return {"direct": direct_ids, "in_comments": comment_ids}

        return self.scrape(url, _list_ids)

    # private functions

    def _parse_roster_table(self, page: BeautifulSoup) -> list[dict]:
        table = page.select_one("#roster")

        if table is None:
            table = self._find_table_in_comments(page, "roster")

        if table is None:
            raise ValueError(
                "Could not locate the roster table. "
                "Run debug_list_tables() to see available table ids."
            )

        rows = []
        for tr in table.select("tbody tr"):
            row = {}
            for cell in tr.select("td, th"):
                stat = cell.get("data-stat")
                if stat:
                    row[stat] = cell.text.strip()
            if row:
                rows.append(row)

        return rows

    @staticmethod
    def _find_table_in_comments(page: BeautifulSoup, id_substring: str):
        for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in comment and id_substring in comment:
                inner = BeautifulSoup(comment, "html.parser")
                found = inner.select_one(f"table[id*={id_substring}]")
                if found is not None:
                    return found
        return None

In [7]:
#GET TEAM ROSTERS:
import time
import random
import pandas as pd

roster_scraper = BaskRefRosterScraper()

all_rosters = []
failed_teams = []

for i, team in enumerate(roster_scraper.TEAM_TRICODES, 1):
    print(f"\r{i}/30: {team}", end="", flush=True)
    try:
        rows = roster_scraper.get_roster(team, 2027)
        all_rosters.extend(rows)
    except Exception as e:
        print(f"\n⚠️ Failed {team}: {type(e).__name__}: {e}")
        failed_teams.append((team, str(e)))
    time.sleep(random.uniform(3, 5))  # polite pacing for a shared-content site

roster_df = pd.DataFrame(all_rosters)
print(f"\n\nTotal players: {len(roster_df)}")
if failed_teams:
    print(f"{len(failed_teams)} teams failed: {failed_teams}")

roster_df.head()

30/30: WAS

Total players: 523


,number,player,pos,height,weight,birth_date,flag,years_experience,college,team,season
0,,Nickeil Alexander-Walker,G,6-5,205,"September 2, 1998",ca CA,7,Virginia Tech,ATL,2027
1,,Jalen Wilson (TW),F,6-6,220,"November 4, 2000",us US,3,Kansas,ATL,2027
2,,Jock Landale,C,6-11,255,"October 25, 1995",au AU,5,Saint Mary's,ATL,2027
3,,Zuby Ejiofor,F,6-9,245,"April 20, 2004",us US,R,"Kansas, St. John's",ATL,2027
4,,Henri Veesaar,C,7-0,227,"March 28, 2004",ee EE,R,"Arizona, UNC",ATL,2027


In [2]:
import unicodedata
def clean_name(s):
    if not isinstance(s, str):
        return s
    try:
        s = s.encode('latin1').decode('utf-8')  # fix mojibake, if present
    except (UnicodeDecodeError, UnicodeEncodeError):
        pass
    normalized = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in normalized if not unicodedata.combining(c))

In [10]:
roster_df['player'] = roster_df['player'].apply(clean_name)
roster_df['player'] = roster_df['player'].str.strip('(TW)')
roster_df

,number,player,pos,height,weight,birth_date,flag,years_experience,college,team,season
0,,Nickeil Alexander-Walker,G,6-5,205,"September 2, 1998",ca CA,7,Virginia Tech,ATL,2027
1,,Jalen Wilson,F,6-6,220,"November 4, 2000",us US,3,Kansas,ATL,2027
2,,Jock Landale,C,6-11,255,"October 25, 1995",au AU,5,Saint Mary's,ATL,2027
3,,Zuby Ejiofor,F,6-9,245,"April 20, 2004",us US,R,"Kansas, St. John's",ATL,2027
4,,Henri Veesaar,C,7-0,227,"March 28, 2004",ee EE,R,"Arizona, UNC",ATL,2027
...,...,...,...,...,...,...,...,...,...,...,...
518,,AJ Dybantsa,F,6-9,217,"January 29, 2007",us US,R,BYU,WAS,2027
519,,Jamir Watkins,F-G,6-6,210,"July 6, 2001",us US,1,"VCU, Florida State",WAS,2027
520,,Felix Okpara,F,6-11,237,"April 20, 2004",ng NG,R,"Ohio State, Tennessee",WAS,2027
521,,rae Young,G,6-2,164,"September 19, 1998",us US,8,Oklahoma,WAS,2027


In [11]:
roster_df.to_csv('2026-27_NBA_rosters.csv')

In [15]:
#DRAFT CLASSES
from dataclasses import dataclass
from bs4 import BeautifulSoup, Comment


@dataclass
class BaskRefDraftScraper(HTMLScraper):
    """Scrapes a single season's NBA draft class page."""

    base_url: str = "https://www.basketball-reference.com"

    def get_draft_class(self, draft_year: int) -> list[dict]:
        url = f"{self.base_url}/draft/NBA_{draft_year}.html"
        rows = self.scrape(url, self._parse_draft_table)
        for row in rows:
            row['draft_year'] = draft_year
        return rows

    def debug_list_tables(self, draft_year: int) -> dict:
        """
        Diagnostic helper: returns every table id found, both directly
        in the DOM and hidden inside HTML comments. Run this first if
        the real parser can't find the draft table.
        """
        url = f"{self.base_url}/draft/NBA_{draft_year}.html"

        def _list_ids(page: BeautifulSoup) -> dict:
            direct_ids = [t.get("id") for t in page.find_all("table")]
            comment_ids = []
            for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
                if "<table" in comment:
                    inner = BeautifulSoup(comment, "html.parser")
                    comment_ids.extend(t.get("id") for t in inner.find_all("table"))
            return {"direct": direct_ids, "in_comments": comment_ids}

        return self.scrape(url, _list_ids)

    # private functions

    def _parse_draft_table(self, page: BeautifulSoup) -> list[dict]:
        # bref has used both "stats" and "draft" as the id over the years
        table = (
            page.select_one("#stats")
            or page.select_one("#draft")
            or page.select_one("table[id*=draft]")
        )

        if table is None:
            table = self._find_table_in_comments(page, "stats")
        if table is None:
            table = self._find_table_in_comments(page, "draft")

        if table is None:
            raise ValueError(
                "Could not locate the draft table. "
                "Run debug_list_tables() to see available table ids."
            )

        rows = []
        for tr in table.select("tbody tr"):
            # skip repeated in-table header rows
            if tr.get("class") and "thead" in tr.get("class"):
                continue

            row = {}
            for cell in tr.select("td, th"):
                stat = cell.get("data-stat")
                if stat:
                    row[stat] = cell.text.strip()
            if row:
                rows.append(row)

        return rows

    @staticmethod
    def _find_table_in_comments(page: BeautifulSoup, id_substring: str):
        for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in comment and id_substring in comment:
                inner = BeautifulSoup(comment, "html.parser")
                found = inner.select_one(f"table[id*={id_substring}]")
                if found is not None:
                    return found
        return None

In [16]:
import time
import random
import pandas as pd

draft_scraper = BaskRefDraftScraper()

draft_years = list(range(2000, 2027))  # 2017 through 2026 inclusive

all_draft_picks = []
failed_years = []

for i, year in enumerate(draft_years, 1):
    print(f"\r{i}/{len(draft_years)}: {year} draft", end="", flush=True)
    try:
        rows = draft_scraper.get_draft_class(year)
        all_draft_picks.extend(rows)
    except Exception as e:
        print(f"\n⚠️ Failed {year}: {type(e).__name__}: {e}")
        failed_years.append((year, str(e)))
    time.sleep(random.uniform(3, 5))

draft_df = pd.DataFrame(all_draft_picks)
print(f"\n\nTotal picks: {len(draft_df)}")
if failed_years:
    print(f"{len(failed_years)} years failed: {failed_years}")

draft_df.head()

27/27: 2026 draft

Total picks: 1612


,ranker,pick_overall,team_id,player,college_name,seasons,g,mp,pts,trb,...,mp_per_g,pts_per_g,trb_per_g,ast_per_g,ws,ws_per_48,bpm,vorp,draft_year,skip
0,1,1,NJN,Kenyon Martin,Cincinnati,15,757,23134,9325,5159,...,30.6,12.3,6.8,1.9,48.0,.100,0.1,12.1,2000,NaN
1,2,2,VAN,Stromile Swift,LSU,9,547,10804,4582,2535,...,19.8,8.4,4.6,0.5,21.3,.095,-1.6,1.1,2000,NaN
2,3,3,LAC,Darius Miles,,7,446,11730,4507,2190,...,26.3,10.1,4.9,1.9,9.5,.039,-1.0,3.0,2000,NaN
3,4,4,CHI,Marcus Fizer,Iowa State,6,289,6032,2782,1340,...,20.9,9.6,4.6,1.2,2.7,.022,-3.7,-2.6,2000,NaN
4,5,5,ORL,Mike Miller,Florida,17,1032,27812,10973,4376,...,26.9,10.6,4.2,2.6,60.7,.105,0.8,19.8,2000,NaN


In [18]:

draft_df['player'] = draft_df['player'].apply(clean_name)
# # draft_df['player'] = draft_df['player'].str.strip('(TW)')
# draft_df
draft_df = pd.read_csv('NBA_draft_classes_2000-2026.csv')


In [19]:
draft_df

,ranker,pick_overall,team_id,player,college_name,seasons,g,mp,pts,trb,...,mp_per_g,pts_per_g,trb_per_g,ast_per_g,ws,ws_per_48,bpm,vorp,draft_year,skip
0,1,1,NJN,Kenyon Martin,Cincinnati,15,757,23134,9325,5159,...,30.6,12.3,6.8,1.9,48.0,.100,0.1,12.1,2000,NaN
1,2,2,VAN,Stromile Swift,LSU,9,547,10804,4582,2535,...,19.8,8.4,4.6,0.5,21.3,.095,-1.6,1.1,2000,NaN
2,3,3,LAC,Darius Miles,,7,446,11730,4507,2190,...,26.3,10.1,4.9,1.9,9.5,.039,-1.0,3.0,2000,NaN
3,4,4,CHI,Marcus Fizer,Iowa State,6,289,6032,2782,1340,...,20.9,9.6,4.6,1.2,2.7,.022,-3.7,-2.6,2000,NaN
4,5,5,ORL,Mike Miller,Florida,17,1032,27812,10973,4376,...,26.9,10.6,4.2,2.6,60.7,.105,0.8,19.8,2000,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1607,56,56,CHI,Vsevolod Ishchenko,,,,,,,...,,,,,,,,,2026,NaN
1608,57,57,ATL,Narcisse Ngoy,,,,,,,...,,,,,,,,,2026,NaN
1609,58,58,NOP,Jaron Pierre Jr.,SMU,,,,,,...,,,,,,,,,2026,NaN
1610,59,59,MIN,Trey Kaufman-Renn,Purdue,,,,,,...,,,,,,,,,2026,NaN


In [13]:
#ADVANCED

from dataclasses import dataclass
from bs4 import BeautifulSoup, Comment


@dataclass
class BaskRefPlayerStatsScraper(HTMLScraper):
    """Scrapes league-wide per-player stat tables (advanced, shooting, etc.)."""

    base_url: str = "https://www.basketball-reference.com"

    STAT_TYPE_CONFIG = {
        "advanced": {
            "url_suffix": "advanced",
            "table_ids": ["advanced", "advanced_stats"],
        },
        "shooting": {
            "url_suffix": "shooting",
            "table_ids": ["shooting", "shooting_stats"],
        },
    }

    def get_stats(self, season_year: int, stat_type: str) -> list[dict]:
        config = self.STAT_TYPE_CONFIG[stat_type]
        url = f"{self.base_url}/leagues/NBA_{season_year}_{config['url_suffix']}.html"
        return self.scrape(url, lambda page: self._parse_stats_table(page, config["table_ids"]))

    def debug_list_tables(self, season_year: int, stat_type: str) -> dict:
        config = self.STAT_TYPE_CONFIG[stat_type]
        url = f"{self.base_url}/leagues/NBA_{season_year}_{config['url_suffix']}.html"

        def _list_ids(page: BeautifulSoup) -> dict:
            direct_ids = [t.get("id") for t in page.find_all("table")]
            comment_ids = []
            for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
                if "<table" in comment:
                    inner = BeautifulSoup(comment, "html.parser")
                    comment_ids.extend(t.get("id") for t in inner.find_all("table"))
            return {"direct": direct_ids, "in_comments": comment_ids}

        return self.scrape(url, _list_ids)

    def _parse_stats_table(self, page: BeautifulSoup, table_ids: list[str]) -> list[dict]:
        table = None
        for tid in table_ids:
            table = page.select_one(f"#{tid}")
            if table is not None:
                break

        if table is None:
            for tid in table_ids:
                table = self._find_table_in_comments(page, tid)
                if table is not None:
                    break

        if table is None:
            raise ValueError(
                f"Could not locate a stats table with any of ids {table_ids}. "
                "Run debug_list_tables() to see available table ids."
            )

        rows = []
        for tr in table.select("tbody tr"):
            if tr.get("class") and "thead" in tr.get("class"):
                continue
            row = {}
            for cell in tr.select("td, th"):
                stat = cell.get("data-stat")
                if stat:
                    row[stat] = cell.text.strip()
            if row:
                rows.append(row)
        return rows

    @staticmethod
    def _find_table_in_comments(page: BeautifulSoup, id_substring: str):
        for comment in page.find_all(string=lambda t: isinstance(t, Comment)):
            if "<table" in comment and id_substring in comment:
                inner = BeautifulSoup(comment, "html.parser")
                found = inner.select_one(f"table[id*={id_substring}]")
                if found is not None:
                    return found
        return None

import time
import random
import pandas as pd

stats_scraper = BaskRefPlayerStatsScraper()  # from earlier in this conversation

seasons = list(range(2000, 2027))  # 2017 through 2026 inclusive

advanced_data_list = []
failed_pulls = []

for i, season in enumerate(seasons, 1):
    print(f"\r{i}/{len(seasons)}: {season} advanced stats", end="", flush=True)
    try:
        rows = stats_scraper.get_stats(season, 'advanced')
        for row in rows:
            row['season'] = season
        advanced_data_list.extend(rows)
    except Exception as e:
        print(f"\n⚠️ Failed {season}: {type(e).__name__}: {e}")
        failed_pulls.append((season, str(e)))
    time.sleep(random.uniform(3, 5))

advanced_stats_df = pd.DataFrame(advanced_data_list)
print(f"\n\nTotal rows: {len(advanced_stats_df)}")
if failed_pulls:
    print(f"{len(failed_pulls)} seasons failed: {failed_pulls}")

advanced_stats_df.head()

27/27: 2026 advanced stats

Total rows: 16653


,ranker,name_display,age,team_name_abbr,pos,games,games_started,mp,per,ts_pct,...,ows,dws,ws,ws_per_48,obpm,dbpm,bpm,vorp,awards,season
0,1,Michael Finley,26,DAL,SF,82,82,3464,19.2,.522,...,6.4,1.9,8.3,.114,3.0,-0.9,2.1,3.6,"MVP-13,AS",2000
1,2,Gary Payton,31,SEA,PG,82,82,3425,23.6,.535,...,10.5,3.5,13.9,.195,5.8,0.7,6.4,7.3,"MVP-6,DPOY-5,AS,NBA1,DEF1",2000
2,3,Latrell Sprewell,29,NYK,SF,82,82,3276,15.7,.515,...,2.5,4.1,6.6,.097,0.5,0.5,1.0,2.5,,2000
3,4,Kevin Garnett,23,MIN,PF,81,81,3243,23.6,.545,...,6.1,5.6,11.6,.172,4.6,1.7,6.3,6.8,"MVP-2,DPOY-7,AS,NBA1,DEF1",2000
4,5,Shareef Abdur-Rahim,23,VAN,SF,82,82,3223,20.2,.547,...,6.2,2.6,8.8,.132,2.6,-0.4,2.2,3.4,,2000


In [20]:
advanced_stats_df.to_csv('NBA_advanced_stats_2000-2026')

In [24]:
advanced_stats_df

,ranker,name_display,age,team_name_abbr,pos,games,games_started,mp,per,ts_pct,...,ows,dws,ws,ws_per_48,obpm,dbpm,bpm,vorp,awards,season
0,1,Michael Finley,26,DAL,SF,82,82,3464,19.2,.522,...,6.4,1.9,8.3,.114,3.0,-0.9,2.1,3.6,"MVP-13,AS",2000
1,2,Gary Payton,31,SEA,PG,82,82,3425,23.6,.535,...,10.5,3.5,13.9,.195,5.8,0.7,6.4,7.3,"MVP-6,DPOY-5,AS,NBA1,DEF1",2000
2,3,Latrell Sprewell,29,NYK,SF,82,82,3276,15.7,.515,...,2.5,4.1,6.6,.097,0.5,0.5,1.0,2.5,,2000
3,4,Kevin Garnett,23,MIN,PF,81,81,3243,23.6,.545,...,6.1,5.6,11.6,.172,4.6,1.7,6.3,6.8,"MVP-2,DPOY-7,AS,NBA1,DEF1",2000
4,5,Shareef Abdur-Rahim,23,VAN,SF,82,82,3223,20.2,.547,...,6.2,2.6,8.8,.132,2.6,-0.4,2.2,3.4,,2000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16648,579,Colby Jones,23,DET,SG,1,0,7,12.9,.333,...,0.0,0.0,0.0,.124,-3.2,5.7,2.4,0.0,,2026
16649,580,Noa Essengue,19,CHI,PF,2,0,6,-11.9,.000,...,-0.1,0.0,-0.1,-0.417,-20.7,-3.0,-23.7,0.0,,2026
16650,581,Trentyn Flowers,20,CHI,SF,2,0,6,16.7,.667,...,0.0,0.0,0.0,.023,1.7,-2.7,-1.0,0.0,,2026
16651,582,Darius Brown II,26,CLE,SG,1,0,3,-7.7,.000,...,0.0,0.0,0.0,-0.269,-16.3,-2.4,-18.7,0.0,,2026


In [25]:
advanced_stats_df= advanced_stats_df.dropna()
advanced_stats_df= advanced_stats_df[advanced_stats_df['name_display'] != 'League Average']
advanced_stats_df['mp']  = advanced_stats_df['mp'].astype(int)
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].apply(clean_name)
multi_team_labels = {'2TM', '3TM', '4TM', '5TM'}

# For each (player, season) group, check if an aggregate row exists
has_agg = advanced_stats_df.groupby(['name_display', 'season'])['team_name_abbr'].transform(
    lambda s: s.isin(multi_team_labels).any()
)

# Rows to drop: belongs to a group with an aggregate row, AND is itself a single-team row
is_individual_team_row = ~advanced_stats_df['team_name_abbr'].isin(multi_team_labels)
drop_mask = has_agg & is_individual_team_row

advanced_stats_df = advanced_stats_df[~drop_mask].copy()
stats_draft = pd.merge(advanced_stats_df,draft_df[['player','draft_year','pick_overall']].drop_duplicates(),how='left',left_on='name_display',right_on='player')
stats_draft = pd.merge(stats_draft,epm[['player_name','season','tot']],how='left',left_on=['name_display','season'],right_on=['player_name','season'])
stats_draft = stats_draft[stats_draft['mp'] >= 200]
stats_draft['exp'] = stats_draft['season'] - stats_draft['draft_year']
stats_draft.groupby('exp').agg({'mean'})

,ranker,name_display,age,team_name_abbr,pos,games,games_started,mp,per,ts_pct,...,bpm,vorp,awards,season,player,draft_year,pick_overall,player_name,tot,exp
0,1,Michael Finley,26,DAL,SF,82,82,3464,19.2,.522,...,2.1,3.6,"MVP-13,AS",2000,NaN,NaN,NaN,NaN,NaN,NaN
1,2,Gary Payton,31,SEA,PG,82,82,3425,23.6,.535,...,6.4,7.3,"MVP-6,DPOY-5,AS,NBA1,DEF1",2000,NaN,NaN,NaN,NaN,NaN,NaN
2,3,Latrell Sprewell,29,NYK,SF,82,82,3276,15.7,.515,...,1.0,2.5,,2000,NaN,NaN,NaN,NaN,NaN,NaN
3,4,Kevin Garnett,23,MIN,PF,81,81,3243,23.6,.545,...,6.3,6.8,"MVP-2,DPOY-7,AS,NBA1,DEF1",2000,NaN,NaN,NaN,NaN,NaN,NaN
4,5,Shareef Abdur-Rahim,23,VAN,SF,82,82,3223,20.2,.547,...,2.2,3.4,,2000,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13148,461,PJ Hall,23,2TM,C,19,2,214,15.9,.599,...,-2.3,0.0,,2026,NaN,NaN,NaN,PJ Hall,-2.584680,NaN
13149,462,Scotty Pippen Jr.,25,MEM,PG,10,6,212,14.7,.537,...,-2.8,0.0,,2026,NaN,NaN,NaN,Scotty Pippen Jr.,-0.806242,NaN
13150,463,DaRon Holmes,23,DEN,PF,25,6,210,13.4,.692,...,-0.5,0.1,,2026,DaRon Holmes,2024.0,22,NaN,NaN,2.0
13151,464,Yuki Kawamura,24,CHI,PG,18,0,209,11.4,.514,...,-3.7,-0.1,,2026,NaN,NaN,NaN,Yuki Kawamura,-2.078450,NaN


In [3]:
%%sql
SELECT * FROM public.player_season_epm

,id,season,age,seasontype,team_id,team_alias,team_alias_all,player_id,player_name,pos_text,...,drbpct,astpct,topct,stlpct,blkpct,fga_rim_75,fga_mid_75,fg3a_75,fta_75,fga_75
0,1,2009-10,25,2,1610612739,CLE,CLE,2544,LeBron James,SF,...,0.184140,0.398529,0.123346,0.022222,0.020047,7.296080,7.81250,5.124470,10.209200,20.23310
1,2,2009-10,21,2,1610612760,OKC,OKC,201142,Kevin Durant,PF,...,0.178313,0.129432,0.118083,0.017680,0.018987,5.081140,10.45770,4.147630,9.878680,19.68650
2,3,2009-10,27,2,1610612748,MIA,MIA,2548,Dwyane Wade,SG,...,0.110373,0.333777,0.122330,0.026662,0.023617,8.217640,9.62477,3.419320,9.765480,21.26170
3,4,2009-10,24,2,1610612753,ORL,ORL,2730,Dwight Howard,C,...,0.315644,0.086852,0.184885,0.013948,0.060622,7.494460,3.96195,0.096971,11.193200,11.55340
4,5,2009-10,24,2,1610612740,NOH,NOH,101108,Chris Paul,PG,...,0.118421,0.456274,0.135103,0.029575,0.003493,3.182660,8.57934,2.928970,4.335790,14.69100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8109,8110,2025-26,26,2,1610612757,POR,POR,1631451,Javonte Cooke,SG,...,0.185185,0.127273,0.176471,0.029126,0.000000,3.605770,2.88462,8.293270,0.721154,14.78370
8110,8111,2025-26,31,2,1610612756,PHX,PHX,1628502,Nigel Hayes-Davis,SF,...,0.125000,0.054688,0.142857,0.020050,0.009662,0.755668,4.91184,3.022670,0.755668,8.69018
8111,8112,2025-26,28,2,1610612741,CHI,"CHI,PHX",1630208,Nick Richards,C,...,0.251135,0.032520,0.193443,0.006775,0.040842,6.766920,2.61449,0.922761,5.177720,10.30420
8112,8113,2025-26,21,2,1610612742,DAL,"DAL,WAS",1642358,AJ Johnson,SG,...,0.101737,0.154341,0.133929,0.011628,0.006048,7.065220,2.17391,4.425470,3.105590,13.66460


In [4]:
epm['season'] = '20' + epm['season'].str[-2:]
epm['season'] = epm['season'].astype(int)
epm

,id,season,age,seasontype,team_id,team_alias,team_alias_all,player_id,player_name,pos_text,...,drbpct,astpct,topct,stlpct,blkpct,fga_rim_75,fga_mid_75,fg3a_75,fta_75,fga_75
0,1,2010,25,2,1610612739,CLE,CLE,2544,LeBron James,SF,...,0.184140,0.398529,0.123346,0.022222,0.020047,7.296080,7.81250,5.124470,10.209200,20.23310
1,2,2010,21,2,1610612760,OKC,OKC,201142,Kevin Durant,PF,...,0.178313,0.129432,0.118083,0.017680,0.018987,5.081140,10.45770,4.147630,9.878680,19.68650
2,3,2010,27,2,1610612748,MIA,MIA,2548,Dwyane Wade,SG,...,0.110373,0.333777,0.122330,0.026662,0.023617,8.217640,9.62477,3.419320,9.765480,21.26170
3,4,2010,24,2,1610612753,ORL,ORL,2730,Dwight Howard,C,...,0.315644,0.086852,0.184885,0.013948,0.060622,7.494460,3.96195,0.096971,11.193200,11.55340
4,5,2010,24,2,1610612740,NOH,NOH,101108,Chris Paul,PG,...,0.118421,0.456274,0.135103,0.029575,0.003493,3.182660,8.57934,2.928970,4.335790,14.69100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8109,8110,2026,26,2,1610612757,POR,POR,1631451,Javonte Cooke,SG,...,0.185185,0.127273,0.176471,0.029126,0.000000,3.605770,2.88462,8.293270,0.721154,14.78370
8110,8111,2026,31,2,1610612756,PHX,PHX,1628502,Nigel Hayes-Davis,SF,...,0.125000,0.054688,0.142857,0.020050,0.009662,0.755668,4.91184,3.022670,0.755668,8.69018
8111,8112,2026,28,2,1610612741,CHI,"CHI,PHX",1630208,Nick Richards,C,...,0.251135,0.032520,0.193443,0.006775,0.040842,6.766920,2.61449,0.922761,5.177720,10.30420
8112,8113,2026,21,2,1610612742,DAL,"DAL,WAS",1642358,AJ Johnson,SG,...,0.101737,0.154341,0.133929,0.011628,0.006048,7.065220,2.17391,4.425470,3.105590,13.66460


In [12]:
advanced_stats_df = pd.read_csv('NBA_advanced_stats_2000-2026')
draft_df = pd.read_csv('NBA_draft_classes_2000-2026.csv')
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].apply(clean_name)
draft_df['player'] = draft_df['player'].apply(clean_name)
multi_team_labels = {'2TM', '3TM', '4TM', '5TM'}

# For each (player, season) group, check if an aggregate row exists
has_agg = advanced_stats_df.groupby(['name_display', 'season'])['team_name_abbr'].transform(
    lambda s: s.isin(multi_team_labels).any()
)

# Rows to drop: belongs to a group with an aggregate row, AND is itself a single-team row
is_individual_team_row = ~advanced_stats_df['team_name_abbr'].isin(multi_team_labels)
drop_mask = has_agg & is_individual_team_row

advanced_stats_df = advanced_stats_df[~drop_mask].copy()
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].str.lower()
advanced_stats_df['name_display'] = advanced_stats_df['name_display'].str.replace('.', '')
epm['player_name'] = epm['player_name'].str.lower()
epm['player_name'] = epm['player_name'].str.replace('.', '')
draft_df['player'] = draft_df['player'].str.lower()
draft_df['player'] = draft_df['player'].str.replace('.', '')
stats_draft = pd.merge(advanced_stats_df,draft_df[['player','draft_year','pick_overall']].drop_duplicates(),how='left',left_on='name_display',right_on='player')
stats_draft = pd.merge(stats_draft,epm[['player_name','season','tot']],how='left',left_on=['name_display','season'],right_on=['player_name','season'])
stats_draft = stats_draft.dropna(subset='tot')
stats_draft['exp'] = stats_draft['season'] - stats_draft['draft_year']
stats_draft = stats_draft[stats_draft['mp'] >= 100].drop_duplicates()
stats_draft = stats_draft[stats_draft['player'] != 'justin jackson']
stats_draft = stats_draft[stats_draft['player'] != 'marcus thornton']
stats_draft = stats_draft.rename(columns={'tot':'epm'})

# # #WINSORIZE
# for col in ['bpm', 'per', 'ws', 'epm']:
#     lower_q = stats_draft.groupby('season')[col].transform(lambda x: x.quantile(0.05))
#     upper_q = stats_draft.groupby('season')[col].transform(lambda x: x.quantile(0.95))
#
#     stats_draft[col] = stats_draft[col].clip(lower=lower_q, upper=upper_q)

# for col in ['bpm', 'per', 'ws', 'epm']:
#     stats_draft[f'{col}_Q3'] = stats_draft.groupby('season')[col].transform(lambda x: x.quantile(0.75))
#     stats_draft[f'{col}_Q1'] = stats_draft.groupby('season')[col].transform(lambda x: x.quantile(0.25))
#     stats_draft[f'{col}_upper_lim'] = stats_draft[f'{col}_Q3'] + (1.5 * (stats_draft[f'{col}_Q3'] - stats_draft[f'{col}_Q1']))
#     stats_draft[f'{col}_lower_lim'] = stats_draft[f'{col}_Q1'] - (1.5 * (stats_draft[f'{col}_Q3'] - stats_draft[f'{col}_Q1']))
#     stats_draft[f'{col}_outlier'] = np.where((stats_draft[col] > stats_draft[f'{col}_upper_lim']) | (stats_draft[col] < stats_draft[f'{col}_lower_lim']),1,0 )



#MAX MIN SCALE 0-10
stats_draft['zs_bpm'] = stats_draft.groupby('season')['bpm'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
stats_draft['zs_per'] = stats_draft.groupby('season')['per'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
stats_draft['zs_ws'] = stats_draft.groupby('season')['ws'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
stats_draft['zs_epm'] = stats_draft.groupby('season')['epm'].transform(lambda x: (x - x.min()) * 10 /(x.max() - x.min()))
stats_draft = stats_draft[~stats_draft['player'].isna()]
stats_draft = pd.merge(stats_draft, stats_draft[stats_draft['exp'] == 1][['name_display']],how='inner',on='name_display')


stats_draft = stats_draft.sort_values(by=['name_display','season'])



adv_stat_weights = [
(0.25 * stats_draft['zs_epm']) + (0.15 * stats_draft['zs_bpm']) + (0.30 * stats_draft['zs_per']) + (0.30 * stats_draft['zs_ws'])  ,
    (0.35 * stats_draft['zs_epm']) + (0.15 * stats_draft['zs_bpm']) + (0.25 * stats_draft['zs_per']) + (0.25 * stats_draft['zs_ws']),
    (0.55 * stats_draft['zs_epm']) + (0.15 * stats_draft['zs_bpm']) + (0.15 * stats_draft['zs_per']) + (0.15 * stats_draft['zs_ws']),
    (0.55 * stats_draft['zs_epm']) + (0.15 * stats_draft['zs_bpm']) + (0.15 * stats_draft['zs_per']) + (0.15 * stats_draft['zs_ws'])
]
conditions = [
    stats_draft['exp'] == 1,
    stats_draft['exp'] == 2,
    stats_draft['exp'] == 3,
    stats_draft['exp'] >= 4
]
stats_draft['advanced_stat_score'] = np.select(conditions,adv_stat_weights)

stats_draft = stats_draft[stats_draft['exp'] <= 4]
within_var = stats_draft.groupby('name_display')['advanced_stat_score'].var().mean()
between_var = stats_draft.groupby('name_display')['advanced_stat_score'].mean().var()

k_estimate = within_var / between_var
# avg_mp = stats_draft['mp'].median()
# k_minutes = k_estimate * avg_mp
#
# stats_draft['mp_weighted_score'] = stats_draft['advanced_stat_score'] * stats_draft['mp']
player_value = (
    stats_draft.groupby('name_display')
    .agg(total=('advanced_stat_score', 'sum'), seasons=('season', 'count'),mp=('mp','sum'))
    .reset_index()
)

overall_mean = stats_draft['advanced_stat_score'].mean()

player_value['advanced_stat_score'] = (
    (player_value['total'] + overall_mean * 2) / (player_value['seasons'] + 2)
)

player_value = pd.merge(player_value,stats_draft[['player','pick_overall','draft_year']].drop_duplicates(),how='left',left_on='name_display',right_on='player')
player_value

,name_display,total,seasons,mp,advanced_stat_score,player,pick_overall,draft_year
0,aaron gordon,15.642639,4,6867.0,3.737161,aaron gordon,4.0,2014.0
1,aaron holiday,12.120376,4,4460.0,3.150117,aaron holiday,23.0,2018.0
2,aaron nesmith,12.232393,4,5054.0,3.168787,aaron nesmith,14.0,2020.0
3,aaron wiggins,14.764592,4,5478.0,3.590820,aaron wiggins,55.0,2021.0
4,ace bailey,2.287552,1,1988.0,3.022627,ace bailey,5.0,2025.0
...,...,...,...,...,...,...,...,...
669,zach lavine,11.678776,4,6601.0,3.076517,zach lavine,13.0,2014.0
670,zeke nnaji,11.956036,4,2399.0,3.122727,zeke nnaji,22.0,2020.0
671,zhaire smith,2.243288,1,111.0,3.007872,zhaire smith,16.0,2018.0
672,ziaire williams,10.785305,4,4486.0,2.927606,ziaire williams,10.0,2021.0


In [13]:
import numpy as np
from scipy import stats

# 1. Define your sample data points
x = player_value['pick_overall']
y = player_value['advanced_stat_score']

# 2. Transform the independent variable x into ln(x)
ln_x = np.log(x)

# 3. Fit a 1st-degree polynomial (linear line) to the transformed data
# Returns coefficients in the order of [slope (b), intercept (a)]
b, a = np.polyfit(ln_x, y, 1)

# 4. Print the final regression equation
print(f"Logarithmic Regression Equation: y = {a:.2f} + {b:.2f} * ln(x)")

slope, intercept, r_value, p_value, std_err = stats.linregress(ln_x, y)

r_squared = r_value ** 2

print(f"y = {intercept:.2f} + {slope:.2f} * ln(x)")
print(f"R² = {r_squared:.4f}")
print(f"p-value = {p_value:.4g}")
print()

Logarithmic Regression Equation: y = 4.05 + -0.24 * ln(x)
y = 4.05 + -0.24 * ln(x)
R² = 0.1247
p-value = 3.117e-21



In [14]:
import plotly.express as px
import plotly.graph_objects as go
player_value['draft_year'] = player_value['draft_year'].astype(str)

fig = px.scatter(player_value,template='plotly_dark', x="pick_overall", y="advanced_stat_score", color="draft_year",
                 size='mp',text='name_display', width=1200, height=1000,title='Value by Draft Pick no. 2009-2025 Draft Classes',subtitle='based on composite score of player value stats')
x_vals = np.linspace(1, 60, 60)

# 3. Apply your mathematical equation (e.g., y = 2x^2 - 5x + 3)
y_vals = 4.05 + -0.24 * np.log(x_vals)

# 4. Append the equation trace to your Express figure
fig.add_trace(
    go.Scatter(
        x=x_vals,
        y=y_vals,
        mode="lines",
        name="4.05 + -0.24 * ln(x)",
        line=dict(color='#636EFA',width=5)
    )
)
fig.update_layout(showlegend=False)
fig.update_coloraxes(showscale=False)
fig.show()
fig.write_image("value_by_draft_pick_no_2009-2025_draft_classes.png")

In [15]:
player_value

,name_display,total,seasons,mp,advanced_stat_score,player,pick_overall,draft_year
0,aaron gordon,15.642639,4,6867.0,3.737161,aaron gordon,4.0,2014.0
1,aaron holiday,12.120376,4,4460.0,3.150117,aaron holiday,23.0,2018.0
2,aaron nesmith,12.232393,4,5054.0,3.168787,aaron nesmith,14.0,2020.0
3,aaron wiggins,14.764592,4,5478.0,3.590820,aaron wiggins,55.0,2021.0
4,ace bailey,2.287552,1,1988.0,3.022627,ace bailey,5.0,2025.0
...,...,...,...,...,...,...,...,...
669,zach lavine,11.678776,4,6601.0,3.076517,zach lavine,13.0,2014.0
670,zeke nnaji,11.956036,4,2399.0,3.122727,zeke nnaji,22.0,2020.0
671,zhaire smith,2.243288,1,111.0,3.007872,zhaire smith,16.0,2018.0
672,ziaire williams,10.785305,4,4486.0,2.927606,ziaire williams,10.0,2021.0


In [16]:
picks = pd.DataFrame({'pick_overall': range(1, 61, 1)})
picks['value'] = 4.05 + -0.24 * np.log(picks['pick_overall'])
picks.to_csv('draft_pick_no_value.csv')
picks

,pick_overall,value
0,1,4.050000
1,2,3.883645
2,3,3.786333
3,4,3.717289
4,5,3.663735
5,6,3.619978
6,7,3.582982
7,8,3.550934
8,9,3.522666
9,10,3.497380


In [13]:
from dataclasses import dataclass, field
import time
import random
import re
import pandas as pd
import requests
from bs4 import BeautifulSoup


# ============================================================
# TEAM MAPPINGS
# ============================================================

TEAM_ID_MAP = {
    'Hawks': 1, 'Celtics': 2, 'Nets': 3, 'Hornets': 4, 'Bulls': 5,
    'Cavaliers': 6, 'Mavericks': 7, 'Nuggets': 8, 'Pistons': 9, 'Warriors': 10,
    'Rockets': 11, 'Pacers': 12, 'Clippers': 13, 'Lakers': 14, 'Grizzlies': 15,
    'Heat': 16, 'Bucks': 17, 'Timberwolves': 18, 'Pelicans': 19, 'Knicks': 20,
    'Thunder': 21, 'Magic': 22, '76ers': 23, 'Suns': 24, 'Trail Blazers': 25,
    'Kings': 26, 'Spurs': 27, 'Raptors': 28, 'Jazz': 29, 'Wizards': 30,
}

TEAM_TRICODE_MAP = {
    'Hawks': 'ATL', 'Celtics': 'BOS', 'Nets': 'BKN', 'Hornets': 'CHA', 'Bulls': 'CHI',
    'Cavaliers': 'CLE', 'Mavericks': 'DAL', 'Nuggets': 'DEN', 'Pistons': 'DET', 'Warriors': 'GSW',
    'Rockets': 'HOU', 'Pacers': 'IND', 'Clippers': 'LAC', 'Lakers': 'LAL', 'Grizzlies': 'MEM',
    'Heat': 'MIA', 'Bucks': 'MIL', 'Timberwolves': 'MIN', 'Pelicans': 'NOP', 'Knicks': 'NYK',
    'Thunder': 'OKC', 'Magic': 'ORL', '76ers': 'PHI', 'Suns': 'PHX', 'Trail Blazers': 'POR',
    'Kings': 'SAC', 'Spurs': 'SAS', 'Raptors': 'TOR', 'Jazz': 'UTA', 'Wizards': 'WAS',
}

REAL_TRICODES = list(TEAM_TRICODE_MAP.values())


# ============================================================
# SCRAPER
# ============================================================

@dataclass
class FanspoOwnedPicksScraper:
    """Scrapes each team's currently-possessed future draft picks from Fanspo."""

    base_url: str = "https://fanspo.com"
    session: requests.Session = field(default_factory=requests.Session)
    headers: dict = field(default_factory=lambda: {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    })

    def get_owned_picks(self, team_name: str, team_id: int) -> list[dict]:
        team_slug = team_name.replace(' ', '%20')
        url = f"{self.base_url}/nba/teams/{team_slug}/{team_id}/draft-picks"
        resp = self.session.get(url, headers=self.headers, timeout=15)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        return self._parse_owned_picks(soup, team_name)

    def _parse_owned_picks(self, soup: BeautifulSoup, owning_team: str) -> list[dict]:
        tables = soup.find_all('table')
        if len(tables) < 2:
            return []

        # table[0] = team info box (coach/exec/arena/etc), table[1] = incoming picks
        incoming_table = tables[1]
        rows = []

        for tr in incoming_table.select('tbody tr'):
            cells = tr.find_all(['th', 'td'])
            # real pick rows are exactly 5 cells: th=year + 4 td (round, pick_no, source, notes).
            # standalone 1-cell rows are just wrapped notes text, redundant with the row above — skip.
            if len(cells) != 5:
                continue

            year, round_, pick_no, source, notes = [c.get_text(strip=True) for c in cells]
            rows.append({
                'year': int(year),
                'round': int(round_),
                'pick_no': pick_no,
                'owning_team': TEAM_TRICODE_MAP.get(owning_team, owning_team),
                'source_raw': source,
                'notes': notes if notes else None,
            })

        return rows


# ============================================================
# SCRAPE ALL 30 TEAMS
# ============================================================

def scrape_all_teams() -> pd.DataFrame:
    scraper = FanspoOwnedPicksScraper()
    all_picks = []
    failed_teams = []

    for i, (team_name, team_id) in enumerate(TEAM_ID_MAP.items(), 1):
        print(f"\r{i}/30: {team_name}", end="", flush=True)
        try:
            rows = scraper.get_owned_picks(team_name, team_id)
            all_picks.extend(rows)
            if not rows:
                print(f"\n⚠️ No rows parsed for {team_name} — worth checking manually")
        except Exception as e:
            print(f"\n⚠️ Failed {team_name}: {type(e).__name__}: {e}")
            failed_teams.append((team_name, str(e)))
        time.sleep(random.uniform(2, 4))

    print(f"\n\nTotal scraped rows: {len(all_picks)}")
    if failed_teams:
        print(f"{len(failed_teams)} teams failed: {failed_teams}")

    return pd.DataFrame(all_picks)


# ============================================================
# CLASSIFY EACH ROW'S SOURCE TEXT
# ============================================================

def classify_own_pick(source_raw: str, owning_team: str):
    """
    Returns (originating_team, rights_note, condition_type).
    originating_team is None when the source describes a multi-team
    arrangement that can't be resolved to a single origin here.
    """
    source = str(source_raw).strip()

    if source.lower() == 'own':
        return owning_team, None, 'own'

    if source.lower().endswith('swap rights'):
        stripped = source[:-len('swap rights')].strip()
        if stripped in REAL_TRICODES:
            # single-team swap right -- this is STILL owning_team's own pick
            return owning_team, stripped, 'swap_right_single'
        return None, stripped, 'swap_right_multi'

    m = re.match(r'via\s+(.+)', source, re.IGNORECASE)
    if m:
        candidate = m.group(1).strip()
        if candidate in REAL_TRICODES:
            return candidate, None, 'traded'
        return None, candidate, 'multi_team_trade'

    return None, source, 'unrecognized'


# ============================================================
# BUILD THE FULL OWNERSHIP MAP
# ============================================================

def build_full_pick_ownership_map(raw_df: pd.DataFrame, years: range = range(2027, 2034)) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Returns (full_pick_map, unresolvable_multiway_rows).
    full_pick_map: every team's own pick, every year, both rounds --
                   resolved rights_holder, defaulting to the team itself
                   when no scraped row exists at all.
    unresolvable_multiway_rows: picks whose origin involves 2+ teams,
                   needing the separate order-statistic resolver.
    """
    parsed = raw_df.apply(
        lambda r: classify_own_pick(r['source_raw'], r['owning_team']),
        axis=1, result_type='expand'
    )
    raw_df = raw_df.copy()
    raw_df[['originating_team', 'rights_note', 'condition_type']] = parsed

    resolvable = raw_df[raw_df['originating_team'].notna()].copy()
    unresolvable = raw_df[raw_df['originating_team'].isna()].copy()

    skeleton = pd.MultiIndex.from_product(
        [REAL_TRICODES, years, [1, 2]],
        names=['originating_team', 'year', 'round']
    ).to_frame(index=False)

    resolvable_slim = resolvable[
        ['originating_team', 'year', 'round', 'owning_team', 'rights_note', 'condition_type', 'notes']
    ].rename(columns={'owning_team': 'rights_holder'})

    full_pick_map = pd.merge(skeleton, resolvable_slim, how='left', on=['originating_team', 'year', 'round'])
    full_pick_map['rights_holder'] = full_pick_map['rights_holder'].fillna(full_pick_map['originating_team'])
    full_pick_map['condition_type'] = full_pick_map['condition_type'].fillna('own')

    return full_pick_map, unresolvable


# ============================================================
# RUN IT ALL
# ============================================================

raw_df = scrape_all_teams()
raw_df.to_csv('NBA_team_draft_picks.csv', index=False)

full_pick_map, unresolvable_multiway = build_full_pick_ownership_map(raw_df)

print(f"\nFull pick ownership map: {len(full_pick_map)} rows")
print(f"Rows still needing multi-way resolution: {len(unresolvable_multiway)}")

full_pick_map.to_csv('full_pick_ownership_map.csv', index=False)
unresolvable_multiway.to_csv('unresolved_multiway_picks.csv', index=False)

full_pick_map

30/30: Wizardslazers

Total scraped rows: 429

Full pick ownership map: 424 rows
Rows still needing multi-way resolution: 56


,originating_team,year,round,rights_holder,rights_note,condition_type,notes
0,ATL,2027,1,SAS,None,traded,No protections
1,ATL,2027,2,ATL,None,own,None
2,ATL,2028,1,ATL,NaN,own,NaN
3,ATL,2028,2,BKN,None,traded,No protections
4,ATL,2029,1,ATL,None,own,None
...,...,...,...,...,...,...,...
419,WAS,2031,2,LAL,None,traded,No protections
420,WAS,2032,1,WAS,None,own,None
421,WAS,2032,2,LAL,None,traded,No protections
422,WAS,2033,1,WAS,None,own,None


In [15]:
full_pick_map_final = full_pick_map[['originating_team','year','round','rights_holder','notes']]
full_pick_map_final

,originating_team,year,round,rights_holder,notes
0,ATL,2027,1,SAS,No protections
1,ATL,2027,2,ATL,None
2,ATL,2028,1,ATL,NaN
3,ATL,2028,2,BKN,No protections
4,ATL,2029,1,ATL,None
...,...,...,...,...,...
419,WAS,2031,2,LAL,No protections
420,WAS,2032,1,WAS,None
421,WAS,2032,2,LAL,No protections
422,WAS,2033,1,WAS,None


In [16]:
full_pick_map_final.to_csv('NBA_team_draft_picks.csv')

In [14]:
owned_picks_df

,team,year,round,pick_no,source,notes
0,Hawks,2027,1,-,NOP or MIL swap rights,NOP will receive the more favorable of its 202...
1,Hawks,2027,2,-,Own,None
2,Hawks,2028,1,-,UTA or CLE swap rights,UTA will receive the most favorable of its 202...
3,Hawks,2029,1,-,Own,None
4,Hawks,2029,2,-,via CLE,No protections
...,...,...,...,...,...,...
424,Wizards,2031,2,-,IND or MIA swap rights,WAS will receive the more favorable of IND's 2...
425,Wizards,2032,1,-,Own,None
426,Wizards,2032,2,-,via UTA,No protections
427,Wizards,2033,1,-,Own,None


In [3]:
%%sql
SELECT *
FROM public.nba_team_dim

,team_id,team,team_alias
0,1610612737,Atlanta Hawks,ATL
1,1610612738,Boston Celtics,BOS
2,1610612739,Cleveland Cavaliers,CLE
3,1610612740,New Orleans Pelicans,NOP
4,1610612768,New Orleans Hornets,NOH
5,1610612741,Chicago Bulls,CHI
6,1610612742,Dallas Mavericks,DAL
7,1610612743,Denver Nuggets,DEN
8,1610612744,Golden State Warriors,GSW
9,1610612745,Houston Rockets,HOU


In [17]:
team_dim

,team_id,team,team_alias
0,1610612737,Atlanta Hawks,ATL
1,1610612738,Boston Celtics,BOS
2,1610612739,Cleveland Cavaliers,CLE
3,1610612740,New Orleans Pelicans,NOP
4,1610612768,New Orleans Hornets,NOH
5,1610612741,Chicago Bulls,CHI
6,1610612742,Dallas Mavericks,DAL
7,1610612743,Denver Nuggets,DEN
8,1610612744,Golden State Warriors,GSW
9,1610612745,Houston Rockets,HOU


In [23]:
team_dim_picks = pd.merge(team_dim,owned_picks_df,how='left',left_on=team_dim["team"].str.split(" ").str[-1],right_on='team')
team_dim_picks.to_csv('NBA_team_draft_picks.csv')